In [30]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications import MobileNetV2
from keras import Sequential
from keras.layers import Dense,BatchNormalization,Dropout,GlobalAveragePooling2D
from keras.regularizers import L2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
test_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\PlantVillage\test"
train_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\PlantVillage\train"
validation_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\PlantVillage\Validation"

In [3]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 30

In [4]:
train_datagen = ImageDataGenerator(rescale=1/255.0)
val_datagen = ImageDataGenerator(rescale=1/255.0)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    validation_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 14440 images belonging to 15 classes.
Found 3089 images belonging to 15 classes.


In [5]:
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False,weights="imagenet")
base_model.trainable = False

In [6]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(64,activation="relu"),
    Dropout(0.2),
    Dense(32,activation="relu"),
    Dropout(0.2),
    Dense(train_generator.num_classes, activation="softmax")
])

In [7]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

In [11]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

os.makedirs("backend/Models", exist_ok=True)

model.save("backend/Models/plant_disease_model.keras")

Epoch 1/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 332s 1s/step - accuracy: 0.6499 - loss: 1.1167 - val_accuracy: 0.8226 - val_loss: 0.5757
Epoch 2/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - accuracy: 0.7898 - loss: 0.6637 - val_accuracy: 0.8725 - val_loss: 0.3881
Epoch 3/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.8380 - loss: 0.5160 - val_accuracy: 0.8938 - val_loss: 0.3335
Epoch 4/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 320s 1s/step - accuracy: 0.8657 - loss: 0.4217 - val_accuracy: 0.9051 - val_loss: 0.3087
Epoch 5/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.8787 - loss: 0.3806 - val_accuracy: 0.9107 - val_loss: 0.2906
Epoch 6/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.8963 - loss: 0.3327 - val_accuracy: 0.9197 - val_loss: 0.2712
Epoch 7/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 325s 1s/step - accuracy: 0.9131 - loss: 0.2800 - val_accuracy: 0.9064 - val_loss: 0.2927
Epoch 8/30
226/226 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.9179 - loss: 0.2692 - val_accu

In [32]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    label_mode="int",
    shuffle=False
)

Found 3109 files belonging to 15 classes.


In [33]:
class_names = test_dataset.class_names

In [34]:
test_dataset = test_dataset.map(
    lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)
)

In [35]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [26]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4%}")

98/98 ━━━━━━━━━━━━━━━━━━━━ 56s 524ms/step - accuracy: 0.9009 - loss: 0.4774
Test Loss: 0.4774
Test Accuracy: 90.0933%


In [37]:
y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot       0.99      0.97      0.98       151
                     Pepper__bell___healthy       0.98      0.99      0.99       223
                      Potato___Early_blight       0.99      0.97      0.98       150
                       Potato___Late_blight       0.92      0.91      0.92       150
                           Potato___healthy       1.00      0.58      0.74        24
                      Tomato_Bacterial_spot       0.94      0.95      0.95       320
                        Tomato_Early_blight       0.66      0.79      0.72       150
                         Tomato_Late_blight       0.89      0.94      0.91       287
                           Tomato_Leaf_Mold       0.78      0.95      0.86       144
                  Tomato_Septoria_leaf_spot       0.95      0.76      0.84       267
Tomato_Spider_mites_Two_spotted_spider_mite       0.76      0.93

In [38]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

[[146   4   0   0   0   0   0   0   0   0   0   0   1   0   0]
 [  1 221   0   0   0   0   0   0   0   0   0   0   1   0   0]
 [  0   0 146   2   0   0   1   0   0   1   0   0   0   0   0]
 [  0   0   1 137   0   1   4   4   1   1   0   0   1   0   0]
 [  0   0   0   9  14   0   0   0   0   0   0   0   0   0   1]
 [  0   0   0   0   0 305   5   2   1   3   0   0   4   0   0]
 [  0   0   0   0   0   4 119  11   7   2   4   0   2   1   0]
 [  0   0   0   0   0   0  13 269   2   0   1   0   1   1   0]
 [  0   0   0   0   0   0   4   1 137   0   0   0   1   1   0]
 [  0   0   1   0   0  12  13   4  14 203   6   3   6   5   0]
 [  0   0   0   1   0   0   1   2   4   1 235   0   4   3   1]
 [  0   0   0   0   0   1  20   6   4   3  44 123   7   3   1]
 [  0   0   0   0   0   1   0   1   1   0   4   0 475   0   0]
 [  0   0   0   0   0   0   0   0   3   0   0   0   1  53   0]
 [  0   0   0   0   0   0   0   1   1   0  14   4   0   2 218]]
